# VRP 02 — Long ATM straddle, held to settlement

**Branch:** Volatility Risk Premium · **Discussion:** `discussions/Volatility_Risk_Premium/007_claude_long_straddle_preregistration.md` · **Code:** `src/data_engineering/modeling/vrp_long_straddle.py`

Does a **buyer** of the 0DTE ATM straddle, paying the ask and the taker fee, recover the cost at settlement on average? No hedge: the spread is paid once and settlement is cash at the exchange delivery price.

Pre-registered (fixed in the module before this notebook was first run on real data):

- Entries every full hour with 2h ≤ TTE ≤ 20h; strike nearest the forward with both legs two-sided.
- $R = \dfrac{|S_T-K|}{\text{ask}_C+\text{ask}_P+\text{fees}} - 1$ (coin premium valued at $S_T/F$ for inverse families).
- $M$ = mean over sessions of the session-pooled mean $R$; 95% session-bootstrap CI.
- **CI > 0 → LONG_EDGE_CANDIDATE · CI < 0 → NO_LONG_EDGE · otherwise INCONCLUSIVE.**
- Sessions on or after 2026-11-07 (VRP confirmation window) are never loaded.

Environment (optional): `VRP02_OFFLINE=1` skips the delivery-price download; `VRP02_ASSETS=BTC,ETH`; `VRP02_N_BOOT`.

In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 250)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_engineering.modeling import vrp_long_straddle as ls
OFFLINE = os.getenv('VRP02_OFFLINE', '').strip() == '1'
ASSETS = [x.strip().upper() for x in os.getenv('VRP02_ASSETS', '').split(',') if x.strip()]
N_BOOT = int(os.getenv('VRP02_N_BOOT', str(ls.N_BOOT)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Risk_Premium' / '02'
OUT.mkdir(parents=True, exist_ok=True)
print('VRP 02', ls.VERSION, ls.selftest())
print(json.dumps(ls.registration(), indent=2))

## 1. Quotes → straddles

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    have = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
    need = [c for c in ls.CHAIN_COLUMNS if c != 'settlement_currency']
    miss = sorted(set(need) - have)
    if miss:
        raise RuntimeError(f'normalized_option_chain lacks {miss}')
    cols = need + (['settlement_currency'] if 'settlement_currency' in have else [])
    where_assets = ('AND n.underlying IN (' + ','.join(repr(a) for a in ASSETS) + ')') if ASSETS else ''
    chain = con.execute(f'''
        WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
        SELECT {', '.join('n.' + c for c in cols)}
        FROM normalized_option_chain n
        JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
        WHERE n.minutes_to_expiry BETWEEN {ls.MIN_TTE_MIN} AND {ls.MAX_TTE_MIN}
          AND EXTRACT(minute FROM n.state_time) % {ls.ENTRY_GRID_MIN} = 0
          AND n.state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'
          AND abs(ln(n.strike / n.option_forward_price)) < 0.05
          {where_assets}
    ''').df()
finally:
    con.close()
if 'settlement_currency' not in chain:
    chain['settlement_currency'] = None
legs = ls.prepare_legs(chain)
st = ls.select_straddles(legs)
print(f'chain rows {len(chain):,} | legs {len(legs):,} | straddles {len(st):,} | sessions {st.session.nunique()}')
display(st.groupby('underlying').agg(straddles=('K', 'size'), sessions=('session', 'nunique'),
                                     median_spread_share=('spread_share', 'median'),
                                     median_fee_share=('fee_usd', lambda f: float(np.median(f / st.loc[f.index, 'ask_usd'])))).round(4))

## 2. Settlement price: exchange delivery price, approximate TWAP as fallback and check

In [ ]:
sessions = sorted(st.session.unique())
con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    idx = con.execute(f'''
        SELECT state_time, underlying, futures_index_price FROM time_series_feature_state
        WHERE state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'
    ''').df()   # UTC window filtering is done in pandas (independent of the DuckDB session time zone)
finally:
    con.close()
approx = ls.approx_settlement(idx, sessions)
names = sorted({ls.index_name(u, c) for u, c in zip(st.underlying, st.settlement_currency)})
cache = OUT / '02_delivery_prices.csv'
if OFFLINE:
    delivery = pd.read_csv(cache, dtype={'session': str}) if cache.exists() else pd.DataFrame(columns=['index_name', 'session', 'delivery_price'])
else:
    delivery = ls.fetch_delivery_prices(names, cache_csv=cache)
delivery = delivery[delivery.session.isin(sessions)]
x = ls.straddle_pnl(ls.attach_settlement(st, delivery, approx))
print('settlement source:', x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict())
chk = x.drop_duplicates(['session', 'underlying']).dropna(subset=['delivery_price', 'settle_approx'])
if len(chk):
    chk = chk.assign(diff_bps=1e4 * np.log(chk.settle_approx / chk.delivery_price))
    settle_check = chk.groupby('underlying').diff_bps.describe()[['count', 'mean', 'std', 'min', 'max']]
    display(settle_check.round(2))
else:
    settle_check = pd.DataFrame()
    print('no overlap between exchange and approximate settlement')

## 3. Primary decision

In [ ]:
prim = ls.primary(x, n_boot=N_BOOT)
print(json.dumps(prim, indent=2))
print('VERDICT:', prim['verdict'])

## 4. Reported, no decision

In [ ]:
cols = list(ls.REPORT_COLS)
tab_all = ls.report_table(x, None, cols, n_boot=1000)
tab_asset = ls.report_table(x, 'underlying', ['R_net', 'R_gross', 'pnl_long_net_bps', 'move_over_implied', 'spread_share'], n_boot=1000)
tab_tte = ls.report_table(x, 'tte_bucket', ['R_net', 'R_gross', 'move_over_implied', 'spread_share'], n_boot=1000)
tab_src = ls.report_table(x, 'settle_source', ['R_net'], n_boot=1000)
keep = ['group', 'level', 'metric', 'n_straddles', 'n_sessions', 'M', 'ci_low', 'ci_high', 'loso_min', 'loso_max', 'share_sessions_positive']
for t in (tab_all, tab_asset, tab_tte, tab_src):
    display(t[[c for c in keep if c in t]].round(4))

In [ ]:
import matplotlib.pyplot as plt
sm = ls.session_means(x, 'R_net')
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(sm)), sm.values, color=np.where(sm.values > 0, '#3a7d44', '#b33f40'))
ax.axhline(0, color='k', lw=0.8); ax.axhline(prim.get('M', np.nan), color='k', ls='--', lw=0.8)
ax.set_xticks(range(len(sm))); ax.set_xticklabels(sm.index, rotation=90, fontsize=7)
ax.set_ylabel('session mean R (net)'); ax.set_title('Long ATM straddle at the ask, held to settlement')
plt.tight_layout(); plt.show()

## 5. Outputs

In [ ]:
session_asset = (x.groupby(['session', 'underlying'])
                 .agg(n=('R_net', 'size'), R_net=('R_net', 'mean'), R_gross=('R_gross', 'mean'),
                      pnl_long_net_bps=('pnl_long_net_bps', 'mean'), move_over_implied=('move_over_implied', 'mean'),
                      S_T=('S_T', 'first'), settle_source=('settle_source', 'first')).reset_index())
session_asset.to_csv(OUT / '02_session_asset.csv', index=False, float_format='%.6g')
pd.concat([tab_all, tab_asset, tab_tte, tab_src], ignore_index=True).to_csv(OUT / '02_report_tables.csv', index=False, float_format='%.6g')
if len(settle_check):
    settle_check.to_csv(OUT / '02_settlement_check.csv', float_format='%.4g')
(OUT / '02_decision.json').write_text(json.dumps({'primary': prim, 'registration': ls.registration()}, indent=2, default=float))
meta = {'module': '02_long_straddle_settlement', 'version': ls.VERSION, 'offline': OFFLINE, 'assets_filter': ASSETS,
        'n_straddles': int(len(x)), 'sessions': [sessions[0], sessions[-1]] if sessions else [],
        'settle_sources': x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict(),
        'index_names': names}
(OUT / '02_run_metadata.json').write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str))